In [1]:
#Все библиотеки тут:
#для парсинга
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm
import random
import csv

#для препроцессинга
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

#для добавления запросов
import json
import requests
from typing import List, Dict
import uuid
import google.generativeai as genai
from typing import List, Dict
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import BitsAndBytesConfig
import torch
import random

#для обучения

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score
from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA
from sklearn.utils import resample
from scipy.stats import mode
from scipy.sparse import issparse 
from sklearn.naive_bayes import (MultinomialNB, ComplementNB)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score
from sentence_transformers import SentenceTransformer
import torch

#для визуализации


[nltk_data] Error loading punkt: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1032)>
[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1032)>
[nltk_data] Error loading wordnet: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1032)>
[nltk_data] Error loading punkt_tab: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1032)>
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stab

In [ ]:
#начинаем с парсинга данных
#задача: собрать табличку книга-аннотация-сслыка на книгу с сайта издательства Бомбора

timeout = (5, 20)
delay_min = 0.5
delay_max = 1.5

books_data = []

for i in tqdm(range(len(book_links))):
    try:
        time.sleep(random.uniform(delay_min, delay_max))
        page = requests.get(book_links[i], timeout=timeout)
        
        if page.status_code != 200:
            books_data.append(['Ошибка загрузки', 'Ошибка загрузки', book_links[i]])
            continue
        
        soup = BeautifulSoup(page.text, 'html.parser')
        
        # Поиск заголовка (пробуем разные варианты)
        title_selectors = [
            'h1.book-overview_title.h2',
            'h1.book-overview__title',
            'h1[class*="title"]',
            'h1'
        ]
        
        title = None
        for selector in title_selectors:
            elem = soup.select_one(selector)
            if elem:
                title = elem.text.strip()
                break
        
        # Поиск описания
        annotation_selectors = [
            '.book-overview_text',
            '.book-overview__text',
            '[class*="annotation"]',
            '[class*="description"]',
            '.book-description'
        ]
        
        annotation = None
        for selector in annotation_selectors:
            elem = soup.select_one(selector)
            if elem:
                annotation = elem.text.strip()
                break
        
        # Если описание не найдено, ищем длинный текст
        if not annotation:
            for p in soup.find_all(['p', 'div']):
                text = p.text.strip()
                if len(text) > 100 and not p.find_parent(['header', 'footer', 'nav']):
                    annotation = text
                    break
        
        books_data.append([
            title if title else 'Заголовок не найден',
            annotation if annotation else 'Описание не найдено',
            book_links[i]
        ])
        
        # Логирование каждых 100 книг (#подсказал дипсик, чтобы отслеживать прогресс 
        # *большой объем и парсинг слетал в ошибку много раз до этого)
        if i % 100 == 0 and i > 0:
            print(f"Обработано {i} книг, успешно: {sum(1 for b in books_data if b[0] != 'Ошибка загрузки')}")
        
    except Exception as e:
        print(f"Ошибка {book_links[i]}: {e}")
        books_data.append(['Ошибка', f'Ошибка: {str(e)}', book_links[i]])
        continue


  1%|          | 101/17687 [04:18<9:27:08,  1.93s/it] 

Обработано 100 книг, успешно: 101


  1%|          | 201/17687 [07:54<8:24:32,  1.73s/it] 

Обработано 200 книг, успешно: 201


  2%|▏         | 301/17687 [11:15<10:17:47,  2.13s/it]

...

100%|█████████▉| 17601/17687 [10:12:41<02:55,  2.04s/it]

Обработано 17600 книг, успешно: 17586


100%|█████████▉| 17651/17687 [10:14:25<02:56,  4.91s/it]

Ошибка https://www.litres.ru/book/dzhi-na-yang/podderzhivat-a-ne-vospityvat-dat-oporu-rebenku-chtoby-on-vyro-73171958/?lfrom=452226304: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


100%|██████████| 17687/17687 [10:15:38<00:00,  2.09s/it]


In [ ]:
#собираем все в табличку через пандас

books_df = pd.DataFrame(books_data)
books_df.columns = ['title', 'annotation', 'link']
books_df

In [ ]:
# Сохраняем результаты на устройство
books_df.to_csv('/Users/uliaocheretina/Desktop/Комп лингв/ML Practice/FINAL_PROJECT_ML/books_data.csv', index=False, encoding='utf-8-sig')

In [2]:
#читаем табличку уже с устройства и сохраняем в переменную
books_df = pd.read_csv('books_data.csv', encoding='utf-8') 

In [3]:
books_df #проверка

,Название,Описание,Ссылка
0,"Любовь, которая убивает. Как распознать психол...","Эта книга для вас, если:\n - в отношениях вы ...",https://bombora.ru/book/185922/
1,"Python все в одном для чайников, 3-е издание","Все, что вам нужно знать, чтобы начать програм...",https://bombora.ru/book/179927/
2,Дневник Бекона из Роблокс. Новый мир! Книга 9,Присоединяйтесь к Билли Блоксу в захватывающем...,https://bombora.ru/book/186999/
3,Комплект из 3-х книг: Зигмунд Фрейд. Классика ...,Откройте для себя психоанализ с новым комплект...,https://bombora.ru/book/186910/
4,Под звездным дождем,Старшеклассница Сонька приезжает к бабушке в д...,https://bombora.ru/book/186909/
...,...,...,...
17682,Сопрано. Закулисная история легендарного сериала,"Подходит для смартфонов, планшетов на Android,...",https://www.litres.ru/book/maykl-imperioli/sop...
17683,Женщина. Тело. Как простые движения запускают ...,"Изменить жизнь к лучшему, начать «с понедельни...",https://bombora.ru/book/179009/
17684,Женщина. Тело. Как простые движения запускают ...,"Подходит для смартфонов, планшетов на Android,...",https://www.litres.ru/book/zarina-del-mar/zhen...
17685,Истории из Minecraft на английском. Читаем с у...,"Учить английский — это весело, особенно если т...",https://bombora.ru/book/178522/


In [4]:
#создание и привязка лейбла названия книги к аннотации, 
# чтобы далее при финальном обучении название подтягивалось само к аннотации

books_df.columns = ['title', 'annotation', 'link'] #на англ удобнее обращаться к колонкам

# генерация уникального айдишника для каждой книги
books_df['book_id'] = [str(uuid.uuid4()) for _ in range(len(books_df))]

# обновляем таблицу 
books_df.to_csv('/Users/uliaocheretina/Desktop/Комп лингв/ML Practice/FINAL_PROJECT_ML/books_id.csv', index=False, encoding='utf-8')
print(f"Всего книг в базе: {len(books_df)}")

# посмотрела через дип сик, что для верности, лучше еще создать словарь 
# Ключ = ID книги, Значение = название книги и ссылка
book_info_id = dict(zip(books_df['book_id'], books_df['title']))
link_id = dict(zip(books_df['book_id'], books_df['link']))

Всего книг в базе: 17687


In [7]:
#также сохраним в переменную и проверка
books_df_id = pd.read_csv('books_id.csv', encoding='utf-8')
books_df_id

,title,annotation,link,book_id
0,"Любовь, которая убивает. Как распознать психол...","Эта книга для вас, если:\n - в отношениях вы ...",https://bombora.ru/book/185922/,cfe2a54e-d4ed-4d30-b625-c7b1dfcb18b4
1,"Python все в одном для чайников, 3-е издание","Все, что вам нужно знать, чтобы начать програм...",https://bombora.ru/book/179927/,1a7ebad7-1a84-461b-a7fd-ca02a17d9544
2,Дневник Бекона из Роблокс. Новый мир! Книга 9,Присоединяйтесь к Билли Блоксу в захватывающем...,https://bombora.ru/book/186999/,1a5212f7-b8c1-4b62-bff1-84935b6b64d9
3,Комплект из 3-х книг: Зигмунд Фрейд. Классика ...,Откройте для себя психоанализ с новым комплект...,https://bombora.ru/book/186910/,f5313349-d2d7-445e-acb4-c0f6a88a2d94
4,Под звездным дождем,Старшеклассница Сонька приезжает к бабушке в д...,https://bombora.ru/book/186909/,a78fba8c-f852-42c6-9104-69bf6958f6fa
...,...,...,...,...
17682,Сопрано. Закулисная история легендарного сериала,"Подходит для смартфонов, планшетов на Android,...",https://www.litres.ru/book/maykl-imperioli/sop...,635f73d2-652d-4628-a2e3-f0e09e8bcbf1
17683,Женщина. Тело. Как простые движения запускают ...,"Изменить жизнь к лучшему, начать «с понедельни...",https://bombora.ru/book/179009/,fed081ef-c5c5-4625-b17b-5c416e3dcbbd
17684,Женщина. Тело. Как простые движения запускают ...,"Подходит для смартфонов, планшетов на Android,...",https://www.litres.ru/book/zarina-del-mar/zhen...,f11ac5c3-6372-4ec9-96f9-a6d8f70a378b
17685,Истории из Minecraft на английском. Читаем с у...,"Учить английский — это весело, особенно если т...",https://bombora.ru/book/178522/,8d8adf0f-544f-46c0-b920-cf71744aaed5


In [8]:
# и еще одна привязка через словарь именно к аннотации
annotation_id = dict(zip(books_df_id['book_id'], books_df_id['annotation']))
book_info_id1 = dict(zip(books_df_id['book_id'], books_df_id['title']))
# пробуем получить аннотацию по айдишнику:
test_id = books_df_id['book_id'].iloc[90]  #тестовое

print(f"ID: {test_id}")
print(f"Название: {book_info_id1[test_id]}")
print(f"Аннотация: {annotation_id[test_id][:100]}...")

ID: 8dc2d217-f479-43ca-9b06-3f9d832ff5e3
Название: Ежедневник в стиле Хобоничи. Японский тренд. Сочные вишни.
Аннотация: Ежедневник в стиле хобоничи — для тех, кто любит японскую «тихую» продуктивность: когда планирование...


In [ ]:
#добавляем запросы пользователей к нашим данным, которые будут частью информации для Х при обучении: консультация у дип сика, первый опыт в интегрировании llm в код, а не запрос напрямую
#решила сделать микс из готового датасета с запросами с hugging face и синтетического, созданного с помощью llm
#upd: в финальную версию вошли только синтетические запросы, тк не хватило мощности и времени на полный сбор 

#генерим запросы через llm (c подсказкой дип сик)
def synthetic_queries_qwen( #беру квен, тк она бесплатная, неплохая для данной задачи + алибаба в целом показывают неплохие результаты на данный момент работ
    df: pd.DataFrame,
    queries_per_book: int = 4,
    max_retries: int = 3
) -> List[Dict]:
    """
    Генерация синтетических запросов через локальную модель Qwen
    """
    print("\n Генерация синтетических запросов через Qwen...")

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # мы берем qwen (изначально хотелось попробовать поработать с claude или gemini, но клод платный, а джемини не удалось запустить даже с впн (чтобы получить код доступа))
    model_name = "Qwen/Qwen2.5-1.5B-Instruct" #беру модель полегче, тк мое устройство не супер мощное по памяти 

    try:
        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True
        )
        # Настройка 4-битной квантизации (для сохранения памяти устройства и ускорения)
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,                      
            bnb_4bit_quant_type="nf4", 
            bnb_4bit_compute_dtype=torch.float16, 
            bnb_4bit_use_double_quant=True          
        )
        
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            trust_remote_code=True,
            device_map="auto",
            dtype=torch.float16 if device == "cuda" else torch.float32,  # экономия памяти
            low_cpu_mem_usage=True
        )
        print("   Модель загружена")
    except Exception as e:
        print(f"   Ошибка загрузки модели: {e}")
        return []
    
    synthetic_pairs = [] #складываем все сгенерированное
    
    for idx, row in df.iterrows():
        book_id = row['book_id']
        annotation = row['annotation']
        title = row['title']
        
        print(f"    Книга {idx+1}/{len(df)}: {title[:30]}...")

        # Промпт для Qwen (формат как для чат-модели)
        messages = [
            {"role": "system", "content": "Ты помогаешь создать датасет для поисковой системы по книгам. Генерируй только запросы, без пояснений."},
            {"role": "user", "content": f"""Сгенерируй {queries_per_book} реалистичных поисковых запросов на русском языке, которые пользователь мог бы ввести, чтобы найти эту книгу.

Название книги: {title}
Описание книги: {annotation}

Требования:
1. Все запросы должны быть на русском языке
2. Используй разные формулировки: вопросы, утверждения, ключевые слова
3. Не используй название книги в запросах
4. Каждый запрос с новой строки, без нумерации

Сгенерируй запросы:"""}
        ]

        # Применяем шаблон чата Qwen
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        
        # Пробуем сгенерировать с повторными попытками
        for attempt in range(max_retries):
            try:
                # Токенизируем
                inputs = tokenizer([text], return_tensors="pt").to(model.device)
                
                # Генерируем
                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=300,
                        temperature=0.8,
                        do_sample=True,
                        top_p=0.9,
                        pad_token_id=tokenizer.eos_token_id
                    )
                    # Декодируем
                generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
                
                # Извлекаем только сгенерированную часть (после промпта)
                # Ищем начало ответа
                if "Сгенерируй запросы:" in generated_text:
                    generated_text = generated_text.split("Сгенерируй запросы:")[-1]
                elif "assistant" in generated_text:
                    generated_text = generated_text.split("assistant")[-1]
                
                # Разбиваем на строки и чистим
                queries = generated_text.strip().split('\n')
                queries = [q.strip() for q in queries if q.strip()]
                
                # Удаляем возможную нумерацию
                cleaned_queries = []
                for q in queries:
                    q = re.sub(r'^\d+[\.\)]\s*', '', q)
                    q = re.sub(r'^[\-\*]\s*', '', q)
                    q = re.sub(r'^["\']|["\']$', '', q)
                    if q and len(q) > 3:  # фильтруем слишком короткие
                        cleaned_queries.append(q.strip())

                # Берем первые queries_per_book запросов
                for q in cleaned_queries[:queries_per_book]:
                    if q:
                        synthetic_pairs.append({
                            'query': q,
                            'book_id': book_id,
                            'source': 'synthetic_qwen',
                            'book_title': title
                        })
                
                print(f"Сгенерировано {len(cleaned_queries[:queries_per_book])} запросов")
                break  # успешно, выходим из retry


            except Exception as e:
                print(f"Попытка {attempt+1}/{max_retries} ошибка: {e}")
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                else:
                    print(f"Не удалось сгенерировать запросы для '{title}'")

        # Небольшая задержка между запросами
        time.sleep(0.5)
        
        # Очищаем кэш GPU (если используем CUDA)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"\n Сгенерировано синтетических запросов: {len(synthetic_pairs)}")
    return synthetic_pairs

            

In [ ]:
#запуск генерации запроса сразу с моей готовой таблицей (с помощью дип сик, первый опыт запуска генерации через код)

synthetic_pairs = synthetic_queries_qwen(
    df=books_df_id,
    queries_per_book=4
)
            
# Сохраняем результат
if synthetic_pairs:
    book_df_synth = pd.DataFrame(synthetic_pairs)
    book_df_synth.to_csv('/Users/uliaocheretina/Desktop/Комп лингв/ML Practice/FINAL_PROJECT_ML/books_synth.csv', index=False, encoding='utf-8')
    books_df_synth = pd.read_csv('books_synth.csv', encoding='utf-8')

    print(f"   - books_synth.csv ({len(synthetic_pairs)} запросов)")

    #примеры генерации
    print("\n примеры:")
    for i in range(min(4, len(synthetic_pairs))):
        pair = synthetic_pairs[i]
        print(f"\n   {i+1}. Запрос: '{pair['query']}'")
        print(f"      → Книга: {pair['book_title']}")
else:
    print("\n Не удалось сгенерировать запросы. Проверьте установку модели.")


#данная часть была сделана в коллабе, тк мое устройсвто недостаотчно мощное, чтобы делать это быстро 
#код тот же, только с сохранением промежуточного результата на гугл диск из-за большого объема данных
#upd: коллаб так же оборвал мне доступ к ускорителю, в итоге сгенерировав запросы на 3000 книг


In [ ]:
#имеем две таблицы, связанные айдишником книги 
books_df_id #табличка с айди и аннтоцией 
books_synth = pd.read_csv('queries_progress.csv', encoding='utf-8') #табличка с айди и запросом

# объединяем таблицы по названию книги, тк при генерации запросов, книгам присвоились новые айдишники
books_full = books_synth.merge(books_df_id[['title', 'annotation', 'link', 'book_id']], left_on='book_title', right_on='title', how='left')

# Проверяем, что аннотации подтянулись
print(f"Колонки: {books_full.columns.tolist()}")

# Смотрим пример
print(books_full[['book_title', 'query', 'annotation']].head(3))

In [10]:
#удаляем ненужные колонки
books_full=books_full.drop(columns=['book_id_x', 'book_title']) #оставляем только те, что были из таблицы с книгами 
books_full=books_full.rename(columns={'book_id_y': 'book_id'})

In [ ]:
#подчистка:
# Удаляем запросы, которые равны "assistant" (баг после генерации)
books_fin_0 = books_full[books_full['query'] != 'assistant']

books_fin_0

In [13]:
#сохраняем таблицу
books_fin_0.to_csv('/Users/uliaocheretina/Desktop/Комп лингв/ML Practice/FINAL_PROJECT_ML/books_join_id_query.csv', index=False, encoding='utf-8-sig')
books_fin = pd.read_csv('books_join_id_query.csv', encoding='utf-8')
#запрос составлен ни на каждую из 17.000 книг, а только на 3000 книг

In [ ]:
#финальная работа с датасетом это добавление рангов, на каждую книгу нужно лейблами добавить отзывы, которые к ней подходят и не подходят (positive/negative)
#upd: изначально было создан дизбаланс классов, но результаты обучения оказались не лучшими, и было принято решение сделать соотношение отзывов 1:1
def pos_neg (df, balance_ratio=1.0, random_seed=42):
    #группируем запрос по книгам
    queries_by_book = df.groupby('book_id')['query'].apply(list).to_dict()
    book_ids = list(queries_by_book.keys())

    #собираем информацию о книгах
    book_info = df.drop_duplicates('book_id').set_index('book_id')[['title', 'annotation', 'link']].to_dict('index')

    #для негативных примеров нам нужно еще раз собрать все запросы
    all_queries = []
    for bid, qs in queries_by_book.items():
        for q in qs:
            all_queries.append({'book_id': bid, 'query': q})


    #правильные запросы
    positive = []
    for book_id, queries in queries_by_book.items():
        info = book_info[book_id]
        for query in queries:
            positive.append({
                'book_id': book_id,
                'title': info['title'],
                'annotation': info['annotation'],
                'link': info['link'],
                'query': query,
                'label': 1
            })
    
    print(f"подходящих примеров: {len(positive)}")

    #неверные запросы
    negative = []
    
    for book_id, queries in queries_by_book.items():
        info = book_info[book_id]
        
        #нужное кол-во
        n_needed = int(len(queries) * balance_ratio)
        #используем запросы других книг
        other_queries = [q for q in all_queries if q['book_id'] != book_id]
        #проверяем достаточно ли запросов
        if len(other_queries) >= n_needed:
            selected = random.sample(other_queries, n_needed)
        else:
        #берем повторки если что
            selected = random.choices(other_queries, k=n_needed)
        
        for item in selected:
            negative.append({
                'book_id': book_id,
                'title': info['title'],
                'annotation': info['annotation'],
                'link': info['link'],
                'query': item['query'], 
                'label': 0
            })
    
    print(f"Негативных примеров: {len(negative)}")

    # объединяем и перемешиваем
    result = pd.DataFrame(positive + negative)
    result = result.sample(frac=1, random_state=random_seed).reset_index(drop=True)
    print(f"\n {len(result)} примеров")
    print(f"Соотношение: 1:{len(negative) / len(positive):.1f}")
    
    return result

In [15]:
books_neg_pos=pos_neg(books_fin, balance_ratio=1.0)

   подходящих примеров: 14289
   Негативных примеров: 14289

 28578 примеров
   Соотношение: 1:1.0


In [ ]:
# проверка
print(books_neg_pos.head())

In [18]:
books_neg_pos.to_csv('/Users/uliaocheretina/Desktop/Комп лингв/ML Practice/FINAL_PROJECT_ML/books_neg_pos.csv', index=False, encoding='utf-8-sig')

books_pn = pd.read_csv('books_neg_pos.csv', encoding='utf-8')

In [ ]:
#приступаем к препроцессингу и токенизации данных

def clean_text(text, language='russian'):
    text = text. lower() #приводим к нижнему регистру
    text = re.sub(r'http\S+|www\S+', ' ', text) #удаляем все похожее на ссылку
    text = re.sub(r'[^a-zа-яё\s]', '', text)  #удаляем все символы кроме букв
    text = re. sub(r'\s+', ' ', text) #убираем лишние пробелы
    tokens=text.split() #токенизируем
    stop_words = set(stopwords.words('russian')) #убираем стоп слова
    return ' '.join(tokens)


#title не трогаем, название нам нужно сырым, чтобы пользователю подгружался оригинал
#очиситим только аннотацию и запросы
books_pn['clean annotation']=books_pn['annotation'].apply(clean_text) #добавляем колонку с чистой аннотацией
books_pn['clean query']=books_pn['query'].apply(clean_text) #добавляем колонку с чистым запросом

books_pn


In [20]:
books_pn.to_csv('/Users/uliaocheretina/Desktop/Комп лингв/ML Practice/FINAL_PROJECT_ML/books_pn.csv', index=False, encoding='utf-8-sig')

In [21]:
#посмотрим итоговую табличку

books_pn.info() #два типа данных числа и текст, построчно все совпадает
print(f'пустые строки:{books_pn.isnull().values.any()}') #пустых строк нет 
print(f'повторяющиеся тексты:{books_pn.duplicated().any()}') #знаю момент с повторками: некоторые из книг подгрузились два раза, тк на сайте они дублируются, только одна ссылкой на бумажный носитель, а другая на литрес
#но решила все равно оставить, вдруг пригодится в будущем

<class 'pandas.DataFrame'>
RangeIndex: 28578 entries, 0 to 28577
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   book_id           28578 non-null  str  
 1   title             28578 non-null  str  
 2   annotation        28578 non-null  str  
 3   link              28578 non-null  str  
 4   query             28578 non-null  str  
 5   label             28578 non-null  int64
 6   clean annotation  28578 non-null  str  
 7   clean query       28578 non-null  str  
dtypes: int64(1), str(7)
memory usage: 81.8 MB
пустые строки:False
повторяющиеся тексты:True


In [22]:
#смотрим дисбаланс классов
books_pn.describe()

#исправили дизбаланс

,label
count,28578.000000
mean,0.500000
std,0.500009
min,0.000000
25%,0.000000
50%,0.500000
75%,1.000000
max,1.000000


In [ ]:
#в процессе был найден еще мусор, убираем
df_clean = books_pn[~books_pn['clean annotation'].str.contains('открывается в программе adobe reader', case=False, na=False)]

df_clean = books_pn[~books_pn['link'].str.contains('litres.ru', case=False, na=False)]


In [ ]:
#было принято попробовать использовать модель логистической регрессии с дальнейшим ранжированием :
#соотвтетсвенно в X мы отправим совмещенные запрос пользователя и аннотацию
#а в у пойдет сам лейбл соответствия: то есть праивльный ответ подходит ли запрос пользователя к аннотации -> если аннотация соотвносится с запросом то к ней по айдишнику подтянется название
#то есть мы обучим модель понимать, подходит ли запрос к аннотации или нет, и на этйо основе уже будет подтягиваться книга 

df_clean['text'] = df_clean['clean query'] + ' [SEP] ' + df_clean['clean annotation'] #объединим колонки

In [29]:
df_clean.to_csv('/Users/uliaocheretina/Desktop/Комп лингв/ML Practice/FINAL_PROJECT_ML/books_pn_text.csv', index=False, encoding='utf-8-sig')

In [30]:
#переходим к распределению материалов
#передаем сразу очищенный текст

train_df, test_df = train_test_split( #определим, что у нас есть материал для для трейна и для теста
    df_clean,
    test_size=0.2,
    random_state=42,
    stratify=df_clean['label']   
)

In [ ]:
#определяем Х и у
X=df_clean['text'].tolist()
y=df_clean['label'].values #таргет уже разбит на бинарник (1, 0), не нуждается в доп кодировании

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) #делим на выборки

In [32]:
#извлечение вектора из текстовых значений
vectorizer=TfidfVectorizer(max_features=5000) #определяем модель векторизации

X_train_vec=vectorizer.fit_transform(X_train) #обучать будем на трейн (тк работаем с обучением с учителем)
X_test_vec=vectorizer.transform(X_test) #тестовую только векторизуем 

print(X_train_vec.shape) #проверяем размерность
print(X_test_vec.shape)

(22862, 5000)
(5716, 5000)


In [33]:
print(type(X_train_vec))

<class 'scipy.sparse._csr.csr_matrix'>


In [34]:
#масштабируем признаки для более точных результатов 
scaler = StandardScaler(with_mean=False)  
X_train_sc = scaler.fit_transform(X_train_vec)
X_test_sc = scaler.transform(X_test_vec)

In [35]:
#обучение модели: берем метод классификации 
#нас интересуем LR

#Логистическая регрессия (баланс классов не нужен, они уже сбалансированы)
model1=LogisticRegression(random_state=42, max_iter=1000)

model1.fit(X_train_sc, y_train)

y_predict1=model1.predict(X_test_sc)

In [36]:
#посмотрим результаты 

print('Логистическая регрессия')
print(classification_report(y_test, y_predict1))

Логистическая регрессия
              precision    recall  f1-score   support

           0       0.42      0.42      0.42      2864
           1       0.41      0.41      0.41      2852

    accuracy                           0.41      5716
   macro avg       0.41      0.41      0.41      5716
weighted avg       0.41      0.41      0.41      5716



In [41]:
#ПРИМЕЧАНИЕ: при обучении модели, признаки оказались ужасными, пробуем улучшить результат посредством обучения на эмбеддингах а не векторах
#пробуем модель фрида мини, одна из лучших и небольших в области работы с русским языком

model_name = "sergeyzh/rubert-mini-frida"
embedder = SentenceTransformer(model_name)

if torch.cuda.is_available():
    embedder = embedder.to('cuda')
    device = "GPU"
else:
    device = "CPU"

print(f"   Модель загружена: {model_name}")


Loading weights: 100%|██████████| 119/119 [00:00<00:00, 6408.20it/s]


   Модель загружена: sergeyzh/rubert-mini-frida


In [ ]:
#создание эмбеддингов
#ПРИМЕЧАНИЕ: не хватило мощности компьютера, перешла в коллаб для выполнения этого шага в облаке
X_train_emb = embedder.encode(
    X_train,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
    precision='float32' 
)

X_test_emb = embedder.encode(
    X_test,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
    precision='float32' 
)

In [ ]:
scaler = StandardScaler()
X_train_sc_em = scaler.fit_transform(X_train_emb)
X_test_sc_em = scaler.transform(X_test_emb)

In [ ]:
model3=LogisticRegression(random_state=42, max_iter=1000)

model3.fit(X_train_sc_em, y_train)

y_predict3=model3.predict(X_test_sc_em)

In [ ]:
print('Логистическая регрессия с балансом классов на эмбеддингах')
print(classification_report(y_test, y_predict3))

In [ ]:
#далее вторая часть в коллабе "Frida"